# EDA na Prática

Nas aulas 9 e 10 vimos os quatro princípios da EDA (estrutural, distribucional, relacional e detecção de outliers/anomalias), como tratar dados ausentes e outliers, e como fazer análise descritiva, bivariada e multivariada. Hoje é a hora de **aplicarmos na prática**. Durante a primeira hora da aula, serão apresentadas as técnicas no dataset *Heart Disease* (UCI) e depois **cada aluno** irá aplicar as técnicas no dataset que foi escolhido semana passada.

*Relembrando*: a EDA não é uma linha reta

A EDA costuma ser apresentada como uma sequência de etapas (estrutura → tratamento → descrição → relação), mas na prática ela é **iterativa**: às vezes tratamos um outlier, olhamos a estatística descritiva de novo, e percebemos que precisa voltar atrás. Hoje vamos deixar isso explícito: olhamos a estatística descritiva **antes** de tratar dados ausentes/outliers (pra saber *como* tratar) e **depois** (pra confirmar o efeito do tratamento).

## Como funcionará a atividade

- **Duração:** ~1h40min
- **Formato:** individual
- **Passo 1:** acompanhem a demonstração no dataset Heart Disease.
- **Passo 2:** importem o dataset que vocês escolheram e enviaram anteriormente.
- **Passo 3:** repliquem o mesmo ciclo no dataset escolhido, respondendo às perguntas guiadas.
- **Uso de IA:** permitido, conforme o Plano de Ensino — documentem na seção final.

## Setup de Bibliotecas e Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## Carregando o Dataset

O arquivo utilizado é o **Used Cars Dataset**. Para facilitar a entrega no Google Colab, a célula abaixo tenta encontrar o CSV e, caso ele não esteja disponível, permite fazer o upload do arquivo.

O dataset possui informações sobre veículos usados, como ano, quilometragem, combustível, transmissão, número de donos, motor, potência e preço.

In [ ]:
import os

CAMINHOS = [
    "/content/used_cars_data (1).csv",
    "used_cars_data (1).csv",
    "/content/used_cars_data.csv",
    "used_cars_data.csv"
]

arquivo = next((c for c in CAMINHOS if os.path.exists(c)), None)

if arquivo is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        arquivo = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "CSV não encontrado. Coloque 'used_cars_data (1).csv' na mesma pasta do notebook."
        )

df = pd.read_csv(arquivo)
print("Arquivo utilizado:", arquivo)
df.head()

# 1. Princípio Estrutural

Antes de analisar os dados, precisamos entender a estrutura da base: quantas observações existem, quantas variáveis estão disponíveis, quais são os tipos de dados e se existem valores duplicados ou ausentes.

### Dimensão e tipos de dados

In [ ]:
print("Número de linhas:", df.shape[0])
print("Número de colunas:", df.shape[1])

print("\nTipos das variáveis:")
display(df.dtypes)

print("\nInformações gerais:")
df.info()

In [ ]:
df.describe(include="all").T

### Duplicadas

Vamos verificar se existem registros completamente duplicados.

**Justificativa:** se não houver duplicadas, não é necessário excluir nenhuma linha nessa etapa. A remoção automática de registros sem duplicação seria uma alteração sem justificativa.

In [ ]:
duplicadas = df.duplicated().sum()
print("Linhas duplicadas:", duplicadas)

### Identificador

A coluna `S.No.` funciona como um identificador sequencial do registro e não representa uma característica do veículo. Por isso, ela não será utilizada nas análises estatísticas.

**Justificativa para a remoção:** manter um identificador numérico nas correlações e estatísticas poderia gerar interpretações sem significado sobre o comportamento dos carros.

In [ ]:
if "S.No." in df.columns:
    df = df.drop(columns=["S.No."])

df.head()

### Separando marca e modelo

A coluna `Name` contém o nome completo do veículo. Para facilitar algumas análises, vamos criar duas novas variáveis:

- `Brand`: marca do veículo;
- `Model`: restante do nome, representando o modelo.

Algumas marcas possuem mais de uma palavra, como **Land Rover**, por isso esse caso será tratado separadamente.

In [ ]:
df["Name"] = df["Name"].astype("string")

def separar_marca(nome):
    if pd.isna(nome):
        return pd.NA, pd.NA

    texto = str(nome).strip()

    if texto.startswith("Land Rover "):
        return "Land Rover", texto[len("Land Rover "):]

    partes = texto.split(maxsplit=1)

    if len(partes) == 1:
        return partes[0], pd.NA

    return partes[0], partes[1]

df[["Brand", "Model"]] = df["Name"].apply(
    lambda x: pd.Series(separar_marca(x))
)

df[["Name", "Brand", "Model"]].head(10)

# 2. Dados Faltantes

Agora vamos verificar quais variáveis possuem informações ausentes.

Além dos valores realmente ausentes (`NaN`), o dataset também possui algumas representações textuais de ausência, como `null`. Por isso, essas representações precisam ser tratadas antes da análise.

In [ ]:
print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False))

print("\nPorcentagem de valores ausentes:")
display((df.isna().mean() * 100).sort_values(ascending=False).round(2))

### Representações de ausência

In [ ]:
df = df.replace(["null", "NULL", "Null", "nan", "NaN", ""], np.nan)

print("Valores ausentes após padronização:")
display(df.isna().sum().sort_values(ascending=False))

### Analisando os valores ausentes

A maior quantidade de ausência está em `New_Price`, seguida por `Price`, `Seats`, `Engine`, `Power` e `Mileage`.

`New_Price` possui uma quantidade muito elevada de valores ausentes. Por isso, não será usada como variável principal nas análises quantitativas. Imputar mais de 80% dessa coluna produziria muitos valores artificiais e poderia distorcer a interpretação.

Já as variáveis numéricas com poucos valores ausentes poderão receber tratamento por mediana quando forem necessárias para a análise.

### Ausências por ano

In [ ]:
cols_ausentes = df.columns[df.isna().any()].tolist()

if cols_ausentes:
    tabela_ano_ausencia = (
        df.groupby("Year")[cols_ausentes]
          .apply(lambda x: x.isna().sum())
    )
    display(tabela_ano_ausencia.head(20))
else:
    print("Não existem colunas com dados ausentes.")

### Tratamento inicial de valores inválidos

Alguns valores não são `NaN`, mas não fazem sentido para a variável:

- `Mileage = 0` não representa um consumo válido;
- `Seats = 0` não representa um veículo com número válido de assentos.

Esses valores serão convertidos para ausência (`NaN`) e tratados posteriormente.

**Justificativa:** é melhor transformar valores claramente inválidos em ausentes do que mantê-los como se fossem medidas reais.

In [ ]:
df["Mileage_value"] = pd.to_numeric(
    df["Mileage"].astype("string").str.extract(r"([0-9]+(?:\.[0-9]+)?)")[0],
    errors="coerce"
)

df["Engine_cc"] = pd.to_numeric(
    df["Engine"].astype("string").str.extract(r"([0-9]+(?:\.[0-9]+)?)")[0],
    errors="coerce"
)

df["Power_bhp"] = pd.to_numeric(
    df["Power"].astype("string").str.extract(r"([0-9]+(?:\.[0-9]+)?)")[0],
    errors="coerce"
)

df.loc[df["Mileage_value"] <= 0, "Mileage_value"] = np.nan
df.loc[df["Seats"] <= 0, "Seats"] = np.nan

print("Ausentes após tratar valores inválidos:")
display(df[["Mileage_value", "Engine_cc", "Power_bhp", "Seats"]].isna().sum())

# 3. Princípio Distribucional

Agora vamos observar como as variáveis quantitativas se distribuem.

Antes do tratamento definitivo, criaremos variáveis derivadas e calcularemos medidas como média, mediana, desvio-padrão, assimetria e curtose.

A variável `New_Price` será convertida para lakh, mas permanecerá fora do conjunto principal de variáveis quantitativas devido ao alto número de valores ausentes.

### Conversão de `New_Price`

In [ ]:
def new_price_to_lakh(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().lower()
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", texto)

    if not match:
        return np.nan

    numero = float(match.group(1))

    if "cr" in texto:
        return numero * 100

    return numero

df["New_Price_lakh"] = df["New_Price"].apply(new_price_to_lakh)

df[["New_Price", "New_Price_lakh"]].head(10)

### Idade do veículo

Para facilitar a interpretação, será criada a variável `Vehicle_Age`.

Como o dataset contém veículos de diferentes anos, a idade será calculada usando o maior ano disponível na própria base como referência.

**Justificativa:** usar o maior ano disponível evita depender da data de execução do notebook e mantém o cálculo reproduzível para esse dataset.

In [ ]:
ano_referencia = int(df["Year"].max())
df["Vehicle_Age"] = ano_referencia - df["Year"]

print("Ano de referência:", ano_referencia)
display(df[["Year", "Vehicle_Age"]].head())

### Variáveis quantitativas

In [ ]:
VARS_QUANT = [
    "Vehicle_Age",
    "Kilometers_Driven",
    "Engine_cc",
    "Power_bhp",
    "Mileage_value",
    "Seats",
    "Price"
]

display(df[VARS_QUANT].describe().T)

### Estatística descritiva, assimetria e curtose

In [ ]:
desc_antes = df[VARS_QUANT].describe().T
desc_antes["skewness"] = df[VARS_QUANT].skew()
desc_antes["kurtosis"] = df[VARS_QUANT].kurtosis()

display(
    desc_antes[
        ["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]
    ]
    .rename(columns={"50%": "median"})
    .round(2)
)

**Discussão:**

A assimetria mostra se a distribuição possui uma cauda mais longa para um dos lados. Valores positivos elevados indicam assimetria à direita.

No dataset, variáveis como `Power_bhp`, `Engine_cc` e principalmente `Price` apresentam valores extremos e tendência à direita. Isso é coerente com uma base que reúne carros populares e carros de maior valor, potência e motorização.

`Kilometers_Driven` também apresenta uma grande dispersão, incluindo registros com quilometragem muito acima da maior parte da amostra.

### Distribuições das variáveis quantitativas

In [ ]:
ncols = 3
nrows = int(np.ceil(len(VARS_QUANT) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, VARS_QUANT):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribuição — {col}")

for ax in axes[len(VARS_QUANT):]:
    ax.remove()

plt.tight_layout()
plt.show()

### Teste de normalidade — Shapiro-Wilk

O teste de Shapiro-Wilk será apresentado como complemento da análise distribucional.

Como o dataset possui milhares de observações, o resultado do teste deve ser interpretado junto com os histogramas e medidas de assimetria, e não isoladamente.

In [ ]:
resultados_shapiro = []

for col in VARS_QUANT:
    serie = df[col].dropna()


    amostra = serie.sample(n=5000, random_state=42) if len(serie) > 5000 else serie

    stat, p = stats.shapiro(amostra)

    resultados_shapiro.append({
        "variavel": col,
        "estatistica": stat,
        "p_valor": p,
        "normal_ao_nivel_5%": p >= 0.05
    })

display(pd.DataFrame(resultados_shapiro).round(4))

### QQ-Plot — preço

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df["Price"].dropna(), dist="norm", plot=ax)
ax.set_title("QQ-Plot — Price")
plt.tight_layout()
plt.show()

### ECDF — preço por tipo de combustível

A ECDF mostra a proporção acumulada de observações e permite comparar as distribuições sem depender da escolha de intervalos de um histograma.

In [ ]:
plt.figure(figsize=(9, 5))
sns.ecdfplot(data=df, x="Price", hue="Fuel_Type")
plt.title("ECDF — Preço por tipo de combustível")
plt.xlabel("Preço (lakh)")
plt.ylabel("Proporção acumulada")
plt.tight_layout()
plt.show()

### Média e mediana do preço

In [ ]:
media_preco = df["Price"].mean()
mediana_preco = df["Price"].median()

print(f"Média do preço: {media_preco:.2f} lakh")
print(f"Mediana do preço: {mediana_preco:.2f} lakh")

plt.figure(figsize=(8, 5))
sns.histplot(df["Price"].dropna(), kde=True)
plt.axvline(media_preco, linestyle="--", label=f"Média = {media_preco:.2f}")
plt.axvline(mediana_preco, linestyle="--", label=f"Mediana = {mediana_preco:.2f}")
plt.title("Preço — média x mediana")
plt.xlabel("Preço (lakh)")
plt.legend()
plt.show()

### Dispersão

In [ ]:
dispersao = df[VARS_QUANT].agg(["var", "std"]).T
dispersao["IQR"] = (
    df[VARS_QUANT].quantile(0.75) -
    df[VARS_QUANT].quantile(0.25)
)

display(dispersao.round(2))

### Boxplots das principais variáveis

In [ ]:
vars_box = [
    "Vehicle_Age",
    "Kilometers_Driven",
    "Engine_cc",
    "Power_bhp",
    "Mileage_value",
    "Price"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, vars_box):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(f"Boxplot — {col}")

plt.tight_layout()
plt.show()

# 4. Tratamento de Dados Ausentes e Outliers

Nesta etapa serão tomadas decisões sobre os dados identificados anteriormente.

A regra utilizada para outliers será o **IQR (Intervalo Interquartil)**:

`limite inferior = Q1 - 1,5 × IQR`

`limite superior = Q3 + 1,5 × IQR`

Um ponto fora desses limites é marcado como possível outlier.

**Importante:** ser identificado pelo IQR não significa automaticamente que o registro é um erro. Um carro muito caro ou muito potente pode ser um caso real.

In [ ]:
def limites_iqr(serie):
    serie = serie.dropna()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

tabela_outliers = []

for col in VARS_QUANT:
    inferior, superior = limites_iqr(df[col])

    qtd = (
        (df[col] < inferior) |
        (df[col] > superior)
    ).sum()

    tabela_outliers.append({
        "variavel": col,
        "limite_inferior": inferior,
        "limite_superior": superior,
        "qtd_outliers": int(qtd),
        "percentual": round(qtd / df[col].notna().sum() * 100, 2)
    })

outliers_df = pd.DataFrame(tabela_outliers)
display(outliers_df.round(2))

### Visualizando os outliers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, vars_box):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(f"Outliers — {col}")

plt.tight_layout()
plt.show()

### Decisão sobre os outliers

Os outliers serão **mantidos na base principal**.

**Justificativa:** o dataset descreve carros reais e não há informação suficiente para afirmar que valores extremos de preço, potência, motor ou quilometragem sejam necessariamente erros de cadastro. Remover esses registros apenas por estarem fora do IQR poderia eliminar veículos legítimos.

A quilometragem máxima, por exemplo, merece atenção especial por ser muito distante da maior parte da amostra, mas sem uma fonte externa não é possível afirmar que seja um erro.

Assim, os outliers serão identificados e documentados, mas não excluídos automaticamente.

### Tratamento dos dados ausentes

Para as variáveis numéricas com poucos valores ausentes, será utilizada a **mediana**.

A mediana é uma escolha adequada aqui porque várias das variáveis apresentam assimetria e valores extremos, situações em que a média pode ser mais influenciada pelos extremos.

`New_Price_lakh` não será imputada na base principal porque aproximadamente 86% dos registros não possuem essa informação. Em vez de criar uma quantidade muito grande de valores artificiais, a variável será mantida apenas para análises específicas.

In [ ]:
df_original = df.copy()

colunas_imputacao = [
    "Mileage_value",
    "Engine_cc",
    "Power_bhp",
    "Seats"
]

df_tratado = df.copy()

for col in colunas_imputacao:
    df_tratado[col] = df_tratado[col].fillna(df_tratado[col].median())

print("Valores ausentes após imputação:")
display(df_tratado[colunas_imputacao].isna().sum())

print("\nPreço sem imputação:")
print("Valores ausentes em Price:", df_tratado["Price"].isna().sum())

### Por que não imputar `Price`?

`Price` é uma das principais variáveis de interesse da análise. Como o valor representa justamente o preço do veículo, substituir preços ausentes pela mediana criaria preços que não foram observados.

Por isso, nas análises que utilizarem `Price`, serão considerados apenas os registros que possuem preço informado.

Essa decisão preserva os valores reais e evita inserir dados artificiais na variável principal.

# 5. Estatística Descritiva — depois do tratamento

Agora vamos repetir a estatística descritiva para verificar como o tratamento dos dados afetou a base.

A comparação é importante porque a EDA não termina quando fazemos a limpeza: precisamos conferir se a decisão tomada produziu uma mudança razoável.

In [ ]:
desc_depois = df_tratado[VARS_QUANT].describe().T
desc_depois["skewness"] = df_tratado[VARS_QUANT].skew()
desc_depois["kurtosis"] = df_tratado[VARS_QUANT].kurtosis()

display(
    desc_depois[
        ["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]
    ]
    .rename(columns={"50%": "median"})
    .round(2)
)

### Comparando antes x depois

In [ ]:
comparacao = pd.DataFrame({
    "skewness_antes": desc_antes["skewness"],
    "skewness_depois": desc_depois["skewness"],
    "kurtosis_antes": desc_antes["kurtosis"],
    "kurtosis_depois": desc_depois["kurtosis"]
})

display(comparacao.round(3))

**Discussão:**

A expectativa é que as medidas das variáveis com poucos valores ausentes sofram pouca alteração, pois a imputação afeta apenas uma pequena parcela dos registros.

Os outliers não foram removidos. Portanto, valores extremos continuam aparecendo nas estatísticas, o que é uma decisão consciente da análise e não um erro de processamento.

### Visualizando antes x depois

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df_original["Price"].dropna(), kde=True, ax=axes[0])
axes[0].set_title("Preço — antes do tratamento")

sns.histplot(df_tratado["Price"].dropna(), kde=True, ax=axes[1])
axes[1].set_title("Preço — depois do tratamento")

plt.tight_layout()
plt.show()

Como `Price` não foi imputado e os outliers foram mantidos, é esperado que sua distribuição permaneça praticamente igual.

A principal diferença do tratamento está nas variáveis que tiveram valores ausentes ou inválidos convertidos e imputados.

# 6. Princípio Relacional

Agora passamos de **"como os dados se distribuem"** para **"como as variáveis se relacionam"**.

A análise bivariada será feita de forma **descritiva e visual**. Os testes estatísticos de hipótese, como Qui-quadrado, Mann-Whitney e Kruskal-Wallis, não serão incluídos porque são opcionais para esta entrega.

## Análise de Variáveis Qualitativas

In [ ]:
VARS_QUALI = [
    "Brand",
    "Location",
    "Fuel_Type",
    "Transmission",
    "Owner_Type"
]

for col in VARS_QUALI:
    print(f"\n### {col}")
    freq = pd.DataFrame({
        "frequencia": df_tratado[col].value_counts(),
        "percentual": df_tratado[col].value_counts(normalize=True) * 100
    })
    display(freq.round(2))

`Model` não foi incluída na tabela principal de frequências porque possui muitas categorias diferentes. Mostrar todas elas em um único gráfico dificultaria a interpretação.

Para `Brand`, também pode ser mais útil visualizar apenas as marcas mais frequentes.

In [ ]:
for col in ["Brand", "Location", "Fuel_Type", "Transmission", "Owner_Type"]:
    plt.figure(figsize=(10, 5))

    if col == "Brand":
        ordem = df_tratado[col].value_counts().head(15).index
        dados = df_tratado[df_tratado[col].isin(ordem)]
        sns.countplot(data=dados, y=col, order=ordem)
    else:
        ordem = df_tratado[col].value_counts().index
        sns.countplot(data=df_tratado, y=col, order=ordem)

    plt.title(f"Frequência — {col}")
    plt.tight_layout()
    plt.show()

### Desbalanceamento entre classes

In [ ]:
from scipy.stats import entropy

def metrica_desbalanceamento(serie):
    contagens = serie.dropna().value_counts()

    if len(contagens) < 2:
        return pd.Series({
            "classe_majoritaria": contagens.index[0],
            "classe_minoritaria": contagens.index[0],
            "razao_desbalanceamento": np.nan,
            "entropia": 0
        })

    razao = contagens.iloc[0] / contagens.iloc[-1]
    probs = contagens / contagens.sum()
    entropia_normalizada = entropy(probs) / np.log(len(probs))

    return pd.Series({
        "classe_majoritaria": contagens.index[0],
        "classe_minoritaria": contagens.index[-1],
        "razao_desbalanceamento": razao,
        "entropia_normalizada": entropia_normalizada
    })

desbalanceamento = pd.DataFrame({
    col: metrica_desbalanceamento(df_tratado[col])
    for col in ["Fuel_Type", "Transmission", "Owner_Type"]
}).T

display(desbalanceamento.round(3))

**Interpretação:**

`Transmission` possui duas categorias e apresenta predominância de carros com transmissão manual.

`Fuel_Type` é bastante desbalanceada porque Diesel e Petrol representam a maior parte dos registros, enquanto CNG, LPG e Electric aparecem em quantidades muito menores.

`Owner_Type` também é desbalanceada, com predominância de veículos classificados como de primeiro dono.

Esse desbalanceamento é importante para análises futuras porque categorias muito pequenas podem produzir comparações pouco representativas.

## Análise Bivariada Descritiva

Como os testes estatísticos bivariados são opcionais, vamos manter apenas comparações visuais e tabelas descritivas.

### Quantitativa × Quantitativa

Uma pergunta relevante para este dataset é:

**Carros mais potentes tendem a apresentar preços maiores?**

Para investigar isso, podemos observar a relação entre `Power_bhp` e `Price`.

In [ ]:
dados_preco_potencia = df_tratado[
    ["Power_bhp", "Price", "Fuel_Type"]
].dropna()

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=dados_preco_potencia,
    x="Power_bhp",
    y="Price",
    hue="Fuel_Type",
    alpha=0.6
)

plt.title("Preço × Potência")
plt.xlabel("Potência (bhp)")
plt.ylabel("Preço (lakh)")
plt.tight_layout()
plt.show()

O gráfico permite observar uma tendência geral de aumento do preço em veículos com maior potência, mas também apresenta bastante dispersão.

Isso mostra que potência é uma característica relacionada ao preço, mas não é o único fator responsável pelo valor de um veículo.

### Quantitativa × Qualitativa

Também podemos comparar o preço entre os tipos de combustível.

Como existem categorias com poucos registros, a interpretação deve considerar o tamanho de cada grupo.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_tratado,
    x="Fuel_Type",
    y="Price"
)

plt.title("Distribuição do preço por tipo de combustível")
plt.xlabel("Tipo de combustível")
plt.ylabel("Preço (lakh)")
plt.tight_layout()
plt.show()

Outra comparação útil é entre os tipos de transmissão.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_tratado,
    x="Transmission",
    y="Price"
)

plt.title("Distribuição do preço por transmissão")
plt.xlabel("Transmissão")
plt.ylabel("Preço (lakh)")
plt.tight_layout()
plt.show()

### Qualitativa × Qualitativa

Sem utilizar testes de hipótese, podemos observar uma tabela de contingência entre `Fuel_Type` e `Transmission`.

A tabela mostra a quantidade de veículos de cada combinação.

In [ ]:
contingencia = pd.crosstab(
    df_tratado["Fuel_Type"],
    df_tratado["Transmission"]
)

display(contingencia)

### Outliers bivariados

Um registro pode não parecer tão extremo quando observamos uma variável isoladamente, mas pode se destacar quando duas variáveis são analisadas juntas.

Vamos observar `Kilometers_Driven` e `Price`.

In [ ]:
dados_joint = df_tratado[
    ["Kilometers_Driven", "Price", "Fuel_Type"]
].dropna()

sns.jointplot(
    data=dados_joint,
    x="Kilometers_Driven",
    y="Price",
    hue="Fuel_Type",
    height=7,
    alpha=0.6
)

plt.show()

A visualização ajuda a identificar veículos que combinam características pouco comuns, como quilometragem muito alta com preço elevado.

Esses pontos não serão removidos automaticamente porque o objetivo desta etapa é identificá-los e compreender seu comportamento.

# 7. Análise Multivariada

Agora vamos observar várias variáveis simultaneamente.

A análise multivariada ajuda a entender o conjunto de relações existentes entre idade, quilometragem, motor, potência, consumo e preço.

### Matriz de correlação — Spearman

In [ ]:
corr_spearman = df_tratado[VARS_QUANT].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_spearman,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Matriz de Correlação — Spearman")
plt.tight_layout()
plt.show()

Como referência para interpretar a intensidade:

- `|r| < 0,3`: relação fraca;
- `0,3 ≤ |r| < 0,7`: relação moderada;
- `|r| ≥ 0,7`: relação forte.

No dataset, é esperado observar uma relação positiva entre potência e preço, enquanto idade e quilometragem tendem a apresentar relação negativa com o preço.

### Principais correlações com o preço

In [ ]:
corr_preco = (
    corr_spearman["Price"]
    .sort_values(ascending=False)
)

display(corr_preco.round(3))

### Pairplot das principais variáveis

In [ ]:
vars_pair = [
    "Vehicle_Age",
    "Kilometers_Driven",
    "Engine_cc",
    "Power_bhp",
    "Price"
]

dados_pair = df_tratado[vars_pair].dropna()

if len(dados_pair) > 1200:
    dados_pair = dados_pair.sample(1200, random_state=42)

sns.pairplot(dados_pair)
plt.show()

### Relação entre preço, idade e quilometragem

A visualização abaixo permite observar simultaneamente três dimensões:

- idade do veículo;
- preço;
- quilometragem.

A transmissão será utilizada para separar os grupos.

In [ ]:
dados_3d = df_tratado[
    ["Vehicle_Age", "Price", "Kilometers_Driven", "Transmission"]
].dropna()

if len(dados_3d) > 2000:
    dados_3d = dados_3d.sample(2000, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=dados_3d,
    x="Vehicle_Age",
    y="Price",
    size="Kilometers_Driven",
    hue="Transmission",
    alpha=0.6,
    sizes=(20, 180)
)

plt.title("Preço × Idade, considerando quilometragem e transmissão")
plt.xlabel("Idade do veículo")
plt.ylabel("Preço (lakh)")
plt.tight_layout()
plt.show()

# 8. Principais Conclusões

A análise exploratória mostra que o dataset possui informações suficientes para investigar características dos veículos e sua relação com o preço.

Os principais pontos observados são:

- A base possui **7.253 registros e 14 colunas originalmente**;
- Não foram encontradas linhas completamente duplicadas;
- `S.No.` foi removida por funcionar apenas como identificador;
- `New_Price` apresenta uma quantidade muito elevada de valores ausentes e, por isso, não foi utilizada como variável quantitativa principal;
- `Price`, `Power_bhp`, `Engine_cc` e `Kilometers_Driven` apresentam distribuições com valores extremos;
- Os outliers foram mantidos porque não há informação suficiente para classificá-los automaticamente como erros;
- Valores ausentes em variáveis numéricas com baixa ausência foram tratados pela mediana;
- O preço apresenta relação positiva com potência e motor e relação negativa com idade e quilometragem, observada de forma descritiva;
- `Fuel_Type`, `Transmission` e `Owner_Type` apresentam diferentes níveis de desbalanceamento;
- Os testes estatísticos da análise bivariada não foram incluídos, conforme permitido nas orientações da atividade.

# 9. Declaração de Uso de IA

**Ferramenta utilizada:** ChatGPT.

**Uso da ferramenta:** auxílio na adaptação da estrutura da aula de EDA para o dataset de carros usados, organização das células do notebook, elaboração de códigos de análise exploratória e explicação das decisões de tratamento dos dados.

